In [3]:

# RUN SAVER 
from pathlib import Path
import json, shutil, datetime, platform, sys
import numpy as np
import pandas as pd

# creating a directory
RUNS_ROOT = Path.home() / "rr_runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
#current time and date
def _ts():
    return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

def make_run_dir(method: str, dataset: str, tag: str = "") -> Path:
    tag = f"_{tag}" if tag else ""
    run_dir = RUNS_ROOT / method / dataset / f"{_ts()}{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def env_info():
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }

def save_run(run_dir: Path,
             per_window_df: pd.DataFrame,
             per_subject_df: pd.DataFrame,
             config: dict,
             splits_path: str | None = None):
    per_window_path = run_dir / "per_window.csv"
    per_subject_path = run_dir / "per_subject.csv"
    config_path = run_dir / "config.json"

    per_window_df.to_csv(per_window_path, index=False)
    per_subject_df.to_csv(per_subject_path, index=False)

    config = dict(config)
    config["environment"] = env_info()
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    if splits_path is not None:
        sp = Path(splits_path)
        if sp.exists():
            shutil.copy2(sp, run_dir / sp.name)

    print("\n✅ Saved run to:", str(run_dir))
    print("  -", per_window_path.name, "| rows:", len(per_window_df))
    print("  -", per_subject_path.name, "| rows:", len(per_subject_df))
    print("  -", config_path.name)
    if splits_path is not None:
        print("  - splits:", Path(splits_path).name, ("OK" if Path(splits_path).exists() else "MISSING"))


# Imports

In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
import pickle
import numpy as np
import pandas as pd

from scipy import signal
from scipy.signal import welch
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from scipy.interpolate import interp1d



# PATHS
PPG_DALIA_RAW_PKL = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Raw_Signal.pkl"        # <-- change this
PPG_DALIA_ANN_PKL = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Annotation.pkl"    # <-- change this
WESAD_RAW_PKL     = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Raw_Signal.pkl"               # <-- change this
WESAD_ANN_PKL     = r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Annotation.pkl"         # <-- change this

# Windowing 
FS_TARGET = 64              # resample to 64 Hz
WIN_SEC   = 32
HOP_SEC   = 2               # RR label every 2s
WIN_SAMPLES = FS_TARGET * WIN_SEC
HOP_SAMPLES = FS_TARGET * HOP_SEC


In [5]:

import gc
def free(*names):
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()

# Loading Datasets


In [4]:
def load_pkl(path):
    assert os.path.exists(path), f"File not found: {path}"
    with open(path, "rb") as f:
        return pickle.load(f)

def quick_inspect(obj, name="OBJ", max_keys=30):
    print(f"\n==== {name} ====")
    print("Type:", type(obj))
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print("Num keys:", len(keys))
        print("Keys (first):", keys[:max_keys])
        # print a few shapes
        for k in keys[:min(10, len(keys))]:
            v = obj[k]
            shp = getattr(v, "shape", None)
            print(f"  {k}: {type(v)} shape={shp}")
    elif isinstance(obj, (list, tuple)):
        print("Len:", len(obj))
        if len(obj) > 0:
            v = obj[0]
            print("First item type:", type(v), "shape:", getattr(v, "shape", None))
    else:
        print("Shape:", getattr(obj, "shape", None))


In [5]:
ppg_dalia_raw = load_pkl(r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Raw_Signal.pkl" )
ppg_dalia_ann = load_pkl(r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\PPG_DaLiA_Annotation.pkl"  )
wesad_raw     = load_pkl(r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Raw_Signal.pkl"  )
wesad_ann     = load_pkl(r"C:\Users\yasmi\OneDrive\Документы\drive-download-20260102T232256Z-1-001\WESAD_Annotation.pkl"    )

quick_inspect(ppg_dalia_raw, "PPG-DaLiA RAW")
quick_inspect(ppg_dalia_ann, "PPG-DaLiA ANN")
quick_inspect(wesad_raw, "WESAD RAW")
quick_inspect(wesad_ann, "WESAD ANN")



==== PPG-DaLiA RAW ====
Type: <class 'numpy.ndarray'>
Shape: (3883, 7, 2048)

==== PPG-DaLiA ANN ====
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (3883, 3)

==== WESAD RAW ====
Type: <class 'numpy.ndarray'>
Shape: (1797, 4, 2048)

==== WESAD ANN ====
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (1797, 3)


# Data

In [6]:
def load_kazemi_raw_pkl(pkl_path: str, channel: int = 0) -> np.ndarray:
    """
    Loads Raw_Signal.pkl.

    Returns:
      X as (N, L) by selecting one channel if the file is (N, C, L).


    """
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)

    X = np.asarray(obj)

    # If already (N, L)
    if X.ndim == 2:
        return X

    # If (N, C, L), select one channel -> (N, L)
    if X.ndim == 3:
        N, C, L = X.shape
        if not (0 <= channel < C):
            raise ValueError(f"channel must be in [0, {C-1}], got {channel}")
        return X[:, channel, :]

    # Sometimes saved as object array of windows -> try vstack
    if X.ndim == 1 and X.dtype == object:
        try:
            X2 = np.vstack(X)
            if X2.ndim == 2:
                return X2
        except Exception:
            pass

    raise ValueError(
        f"Raw signal PKL doesn't look like window matrix (N,L) or (N,C,L). "
        f"Got shape={X.shape}, dtype={X.dtype}."
    )


# Helpers

In [1]:
def zscore(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x)
    return (x - np.mean(x)) / (np.std(x) + eps)

def butter_bandpass(x: np.ndarray, fs: float, low_hz: float, high_hz: float, order: int = 4) -> np.ndarray:
    nyq = 0.5 * fs
    low = max(low_hz / nyq, 1e-6)
    high = min(high_hz / nyq, 0.999999)
    b, a = signal.butter(order, [low, high], btype="bandpass")
    return signal.filtfilt(b, a, x)


NameError: name 'np' is not defined

# SQI

In [8]:
def _match_peaks(peaks_a: np.ndarray, peaks_b: np.ndarray, tol_samp: int) -> Tuple[int, int, int]:
    # returns TP, FP, FN for a→b matching within tol
    peaks_a = np.asarray(peaks_a)
    peaks_b = np.asarray(peaks_b)
    if len(peaks_a) == 0 and len(peaks_b) == 0:
        return 0, 0, 0
    if len(peaks_a) == 0:
        return 0, len(peaks_b), 0
    if len(peaks_b) == 0:
        return 0, 0, len(peaks_a)

    used_b = np.zeros(len(peaks_b), dtype=bool)
    tp = 0
    for pa in peaks_a:
        # find nearest b
        idx = np.argmin(np.abs(peaks_b - pa))
        if not used_b[idx] and abs(peaks_b[idx] - pa) <= tol_samp:
            used_b[idx] = True
            tp += 1

    fp = np.sum(~used_b)  # b peaks not matched
    fn = len(peaks_a) - tp  # a peaks not matched
    return int(tp), int(fp), int(fn)

def compute_sqi(ppg: np.ndarray, fs: float,
                flat_hyst_frac: float = 0.005,
                tol_ms: float = 150.0) -> Tuple[float, np.ndarray, np.ndarray]:
    """
    SQI in [0,1].
    - Flatline detector: small successive changes => "flat"
    - Dual peak detectors: agreement F1-score
    """
    x = np.asarray(ppg).astype(float)
    x = x - np.median(x)

    # Flatline proportion
    # hysteresis threshold is fraction of signal range
    rng = np.percentile(x, 95) - np.percentile(x, 5)
    hyst = max(flat_hyst_frac * (rng + 1e-9), 1e-6)
    flat = (np.abs(np.diff(x, prepend=x[0])) < hyst)
    nonflat_prop = 1.0 - np.mean(flat)

    # Two slightly different peak detectors
    # basic constraints: plausible HR 35-220 bpm => min distance
    min_dist = int(fs * 60.0 / 220.0)  # conservative
    prom_a = 0.25 * np.std(x)
    prom_b = 0.35 * np.std(x)

    peaks_a, _ = signal.find_peaks(x, distance=min_dist, prominence=prom_a)
    peaks_b, _ = signal.find_peaks(x, distance=min_dist, prominence=prom_b)

    tol_samp = int(round((tol_ms / 1000.0) * fs))
    tp, fp, fn = _match_peaks(peaks_a, peaks_b, tol_samp)

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    sqi = float(np.clip(f1 * nonflat_prop, 0.0, 1.0))
    return sqi, peaks_a, peaks_b


# spectral SQI per window

In [9]:
def ppg_sqi_spectral(ppg_win: np.ndarray, fs: float, rr_band_hz: tuple) -> float:
    """
    Very simple SQI: ratio of max power in RR band to total power in [0.05, 8] Hz.
    Higher = more respiration-like structure present (often correlates with usable windows).
    """
    ppg_win = np.asarray(ppg_win).astype(float)
    ppg_win = zscore(ppg_win)
    freqs, pxx = signal.welch(ppg_win, fs=fs, window="hann", nperseg=min(len(ppg_win), int(fs*16)), noverlap=int(fs*8))
    total_band = (freqs >= 0.05) & (freqs <= 8.0)
    rr_band = (freqs >= rr_band_hz[0]) & (freqs <= rr_band_hz[1])
    if not (np.any(total_band) and np.any(rr_band)):
        return np.nan
    total_power = np.trapz(pxx[total_band], freqs[total_band]) + 1e-12
    rr_peak = np.max(pxx[rr_band]) if np.any(rr_band) else 0.0
    return float(rr_peak / total_power)

# Extract RIIV, RIAV, RIFV

In [10]:
def extract_respiratory_induced_variations(ppg: np.ndarray, fs: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns:
      t_peaks (sec), RIIV, RIAV, RIFV (aligned to peak times, irregular sampling)
    Key change vs your version:
      - Use bandpassed signal ONLY to detect peaks/troughs
      - Compute RIIV/RIAV amplitudes from raw detrended signal (keeps respiratory modulation)
    """
    x_raw = np.asarray(ppg).astype(float)
    x_raw = x_raw - np.median(x_raw)

    # Beat detection helper (cardiac band)
    x_det = butter_bandpass(x_raw, fs, low_hz=0.5, high_hz=8.0, order=3)

    # Peak detection (beats)
    min_dist = int(fs * 60.0 / 220.0)
    prom = 0.25 * np.std(x_det)
    peaks, _ = signal.find_peaks(x_det, distance=min_dist, prominence=prom)

    if len(peaks) < 3:
        return np.array([]), np.array([]), np.array([]), np.array([])

    # Trough detection on inverted detection signal
    troughs, _ = signal.find_peaks(-x_det, distance=min_dist)
    troughs_sorted = np.sort(troughs)

    # Pair each peak with the most recent trough before it
    trough_for_peak = []
    for p in peaks:
        idx = np.searchsorted(troughs_sorted, p) - 1
        trough_for_peak.append(troughs_sorted[idx] if idx >= 0 else -1)
    trough_for_peak = np.array(trough_for_peak)

    # keep only peaks that have a valid preceding trough
    mask = trough_for_peak >= 0
    peaks = peaks[mask]
    trough_for_peak = trough_for_peak[mask]

    if len(peaks) < 3:
        return np.array([]), np.array([]), np.array([]), np.array([])

    t_peaks = peaks / fs

    # IMPORTANT: amplitudes from RAW (detrended), not filtered cardiac-band signal
    riiv = x_raw[peaks]                              # intensity variation (peak amplitude)
    riav = x_raw[peaks] - x_raw[trough_for_peak]     # amplitude variation (peak - trough)

    # RIFV: instantaneous heart rate from peak intervals
    ibi = np.diff(t_peaks)           # seconds
    rifv = 1.0 / (ibi + 1e-9)        # Hz
    t_rifv = t_peaks[1:]

    # Align RIIV/RIAV to RIFV timestamps (drop first peak)
    riiv = riiv[1:]
    riav = riav[1:]
    t_peaks = t_rifv

    return t_peaks, riiv, riav, rifv


# AR

In [11]:
def burg_ar(x: np.ndarray, order: int) -> Tuple[np.ndarray, float]:
    """
    Burg algorithm for AR coefficients.
    Returns:
      a: AR polynomial [1, a1, ..., ap]
      e: final prediction error variance
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if order >= n - 1:
        order = max(1, n - 2)

    # Initialize
    ef = x[1:].copy()
    eb = x[:-1].copy()
    a = np.zeros(order + 1, dtype=float)
    a[0] = 1.0
    E = np.dot(x, x) / n

    for k in range(1, order + 1):
        # reflection coefficient
        num = -2.0 * np.dot(eb, ef)
        den = (np.dot(ef, ef) + np.dot(eb, eb)) + 1e-12
        gamma = num / den

        # update AR coeffs
        a_new = a.copy()
        a_new[1:k] = a[1:k] + gamma * a[k-1:0:-1]
        a_new[k] = gamma
        a = a_new

        # update errors
        ef_new = ef + gamma * eb
        eb_new = eb + gamma * ef
        ef, eb = ef_new[1:], eb_new[:-1]

        # update error
        E *= (1.0 - gamma**2)

        if len(ef) < 2 or len(eb) < 2:
            break

    return a, float(max(E, 1e-12))

def ar_amplitude_spectrum(a: np.ndarray, e: float, fs: float, nfft: int = 1024) -> Tuple[np.ndarray, np.ndarray]:
    """
    AR spectrum magnitude on [0, fs/2].
    """
    w = np.linspace(0, np.pi, nfft//2 + 1)
    # A(e^jw)
    jw = np.exp(-1j * w)
    A = np.polyval(a, jw)  # a[0] + a1 z^-1 + ...
    # PSD ~ e / |A|^2 ; we can use amplitude ~ 1/|A|
    amp = 1.0 / (np.abs(A) + 1e-12)
    f = (w / (2*np.pi)) * fs
    return f, amp.real

def fused_rr_from_three_variations(
    t_peaks: np.ndarray,
    riiv: np.ndarray,
    riav: np.ndarray,
    rifv: np.ndarray,
    fs_resamp: float = 4.0,
    orders: Optional[List[int]] = None,
    rr_min_bpm: float = 4.0,
    rr_max_bpm: float = 65.0,
    nfft: int = 1024
) -> Optional[float]:
    """
    Returns RR in brpm, or None if cannot estimate.
    Implements: resample → zscore → multiple AR spectra (Burg) → median fusion → peak in RR band.
    """
    if orders is None:
        orders = list(range(2, 19))  # reasonable default: 2..18

    if len(t_peaks) < 6:
        return None

    # resample irregular signals onto uniform grid
    t0, t1 = float(t_peaks[0]), float(t_peaks[-1])
    if t1 - t0 < 5.0:
        return None

    t_u = np.arange(t0, t1, 1.0 / fs_resamp)
    if len(t_u) < 16:
        return None

    def resamp(ts, xs):
        f = interp1d(ts, xs, kind="linear", fill_value="extrapolate", bounds_error=False)
        return f(t_u)

    x1 = zscore(resamp(t_peaks, riiv))
    x2 = zscore(resamp(t_peaks, riav))
    x3 = zscore(resamp(t_peaks, rifv))

    spectra = []
    for x in (x1, x2, x3):
        # remove mean again to be safe
        x = x - np.mean(x)
        for p in orders:
            if p >= len(x) - 2:
                continue
            a, e = burg_ar(x, p)
            f, amp = ar_amplitude_spectrum(a, e, fs=fs_resamp, nfft=nfft)
            spectra.append(amp)

    if len(spectra) == 0:
        return None

    spectra = np.vstack(spectra)  # (M, F)
    med = np.median(spectra, axis=0)

    # pick peak in RR band
    fmin = rr_min_bpm / 60.0
    fmax = rr_max_bpm / 60.0
    band = (f >= fmin) & (f <= fmax)
    if not np.any(band):
        return None

    idx = np.argmax(med[band])
    f_peak = f[band][idx]
    return float(f_peak * 60.0)


# One-Window baseline estimator

In [12]:
@dataclass
class PimentelConfig:
    fs_ppg: float = 64.0          # unified rate
    win_sec: float = 32.0
    fs_resamp: float = 4.0        
    rr_min_bpm: float = 4.0
    rr_max_bpm: float = 65.0
    sqi_threshold: float = 0.90   
    use_sqi: bool = True
    orders: Optional[List[int]] = None

def estimate_rr_pimentel_window(ppg_win: np.ndarray, cfg: PimentelConfig) -> Tuple[Optional[float], float]:
    """
    Returns (rr_est_brpm or None, sqi).
    """
    x = np.asarray(ppg_win).astype(float)

    sqi, _, _ = compute_sqi(x, fs=cfg.fs_ppg)
    if cfg.use_sqi and sqi < cfg.sqi_threshold:
        return None, sqi

    t_peaks, riiv, riav, rifv = extract_respiratory_induced_variations(x, fs=cfg.fs_ppg)
    rr = fused_rr_from_three_variations(
        t_peaks, riiv, riav, rifv,
        fs_resamp=cfg.fs_resamp,
        orders=cfg.orders,
        rr_min_bpm=cfg.rr_min_bpm,
        rr_max_bpm=cfg.rr_max_bpm
    )
    return rr, sqi


# Evaluation

In [13]:
def mae_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if np.sum(mask) == 0:
        return np.nan, np.nan
    err = y_pred[mask] - y_true[mask]
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    return mae, rmse

def bland_altman_stats(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float, float]:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    d = (y_pred[mask] - y_true[mask])
    bias = float(np.mean(d))
    sd = float(np.std(d, ddof=1)) if len(d) > 1 else np.nan
    loa_low = bias - 1.96 * sd
    loa_high = bias + 1.96 * sd
    return bias, loa_low, loa_high

def run_baseline_on_split(
    X_ppg: np.ndarray,
    y_rr: np.ndarray,
    subject: np.ndarray,
    test_subjects: List[int],
    cfg: PimentelConfig
) -> Dict[str, float]:
    test_mask = np.isin(subject, np.array(test_subjects))
    X_te = X_ppg[test_mask]
    y_te = y_rr[test_mask]

    preds = np.full(len(y_te), np.nan, dtype=float)
    sqis = np.full(len(y_te), np.nan, dtype=float)

    for i in range(len(y_te)):
        rr_hat, sqi = estimate_rr_pimentel_window(X_te[i], cfg)
        sqis[i] = sqi
        if rr_hat is not None:
            preds[i] = rr_hat

    coverage = float(np.mean(np.isfinite(preds)))
    mae, rmse = mae_rmse(y_te, preds)
    bias, loa_low, loa_high = bland_altman_stats(y_te, preds)

    return {
        "N_test": int(len(y_te)),
        "coverage": coverage,
        "MAE": mae,
        "RMSE": rmse,
        "bias": bias,
        "LoA_low": loa_low,
        "LoA_high": loa_high
    }


# Test

In [6]:
def load_kazemi_annotation_pkl(pkl_path: str) -> Dict[str, np.ndarray]:
    """
    Loads *_Annotation.pkl and returns:
      rr      -> (N,)
      subject -> (N,)
      keys    -> column names (for debugging)

    Supports:
      - pandas.DataFrame
      - dict
    """
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)

    # ----------------------------
    # Case 1: pandas DataFrame
    # ----------------------------
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
        cols = list(df.columns)

        # helper: find a column by common aliases (case-insensitive)
        def pick_col(*cands):
            lower = {c.lower(): c for c in cols}
            for cand in cands:
                if cand.lower() in lower:
                    return lower[cand.lower()]
            return None

        rr_col  = pick_col("Reference_RR", "rr", "resp", "respr", "resp_rate", "respiratory_rate", "brpm", "breaths_per_min", "label", "y")
        sid_col = pick_col("patient_id", "subject", "subject_id", "sid", "subj", "subjects", "user", "participant", "pid")


        # fallback: try partial matches
        if rr_col is None:
            for c in cols:
                cl = c.lower()
                if ("rr" == cl) or ("resp" in cl and "rate" in cl) or ("breath" in cl):
                    rr_col = c
                    break

        if sid_col is None:
            for c in cols:
                cl = c.lower()
                if ("subject" in cl) or (cl in ("sid", "pid", "user", "participant")):
                    sid_col = c
                    break

        if rr_col is None or sid_col is None:
            print("Annotation DataFrame columns:", cols)
            print("Head:\n", df.head())
            raise ValueError(
                "Could not locate RR and subject columns in Annotation DataFrame. "
                "See printed columns/head and update alias lists in loader."
            )

        rr = df[rr_col].to_numpy(dtype=float).reshape(-1)
        sid = df[sid_col].to_numpy().reshape(-1)

        return {"rr": rr, "subject": sid, "keys": np.array(cols, dtype=object)}

    # ----------------------------
    # Case 2: dict
    # ----------------------------
    if isinstance(obj, dict):
        keys = list(obj.keys())

        def pick_key(*cands):
            for c in cands:
                if c in obj:
                    return obj[c]
            # case-insensitive fallback
            lower = {k.lower(): k for k in keys}
            for c in cands:
                if c.lower() in lower:
                    return obj[lower[c.lower()]]
            return None

        rr = pick_key("rr", "RR", "resp", "RespR", "y", "y_rr", "labels_rr")
        sid = pick_key("subject", "subject_id", "sid", "subj", "subjects", "user", "participant")

        if rr is None or sid is None:
            print("Annotation dict keys found:", keys)
            raise ValueError("Could not find rr and subject in Annotation dict. See printed keys and update aliases.")

        rr = np.asarray(rr).astype(float).reshape(-1)
        sid = np.asarray(sid).reshape(-1)

        return {"rr": rr, "subject": sid, "keys": np.array(keys, dtype=object)}

    # ----------------------------
    # Otherwise unsupported
    # ----------------------------
    raise ValueError(f"Unsupported Annotation PKL type: {type(obj)}. Expected pandas.DataFrame or dict.")


In [7]:
free(
    "ppg_dalia_raw", "ppg_dalia_ann", "wesad_raw", "wesad_ann",
    "dalia_raw", "wesad_raw",
    "dalia_X", "wesad_X",
    "wesad_rows", "dalia_rows",
    "df_wesad", "df_dalia"
)


In [16]:

# 1) Pick channels found from the sweep:
WESAD_PPG_CH = 1
DALIA_PPG_CH = 0

# 2) Load X from Raw_Signal.pkl (returns (N,L) after selecting channel)
wesad_X = load_kazemi_raw_pkl(WESAD_RAW_PKL, channel=WESAD_PPG_CH)
dalia_X = load_kazemi_raw_pkl(PPG_DALIA_RAW_PKL, channel=DALIA_PPG_CH)

# 3) Load rr + subject from Annotation.pkl (your Annotation is a DataFrame)
wesad_ann = load_kazemi_annotation_pkl(WESAD_ANN_PKL)
dalia_ann = load_kazemi_annotation_pkl(PPG_DALIA_ANN_PKL)

# 4) Make subject IDs safe (int, not float)
wesad_sid = wesad_ann["subject"].astype(int)
dalia_sid = dalia_ann["subject"].astype(int)

wesad_rr = wesad_ann["rr"].astype(float)
dalia_rr = dalia_ann["rr"].astype(float)

# 5) Sanity checks
print("WESAD X:", wesad_X.shape, "| y:", wesad_rr.shape, "| sid:", wesad_sid.shape)
print("DaLiA X:", dalia_X.shape, "| y:", dalia_rr.shape, "| sid:", dalia_sid.shape)

assert wesad_X.shape[0] == len(wesad_rr) == len(wesad_sid), "WESAD: X/y/sid mismatch"
assert dalia_X.shape[0] == len(dalia_rr) == len(dalia_sid), "DaLiA: X/y/sid mismatch"

print("WESAD unique patient_id:", np.unique(wesad_sid))
print("DaLiA unique patient_id:", np.unique(dalia_sid))

# 6) Config (same as you had)
cfg = PimentelConfig(
    fs_ppg=64.0,
    win_sec=32.0,
    fs_resamp=4.0,
    rr_min_bpm=4.0,
    rr_max_bpm=65.0,
    use_sqi=True,
    sqi_threshold=0.90,
    orders=list(range(2, 19))
)

# 7) ✅ Put your subject split here (must match the printed unique patient_id values)
wesad_test_subjects = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # <-- change ONLY this if needed
dalia_test_subjects = [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]  # <-- change ONLY this if needed

# 8) Run baseline
wesad_results = run_baseline_on_split(wesad_X, wesad_rr, wesad_sid, wesad_test_subjects, cfg)
dalia_results = run_baseline_on_split(dalia_X, dalia_rr, dalia_sid, dalia_test_subjects, cfg)

print("Pimentel baseline — WESAD:", wesad_results)
print("Pimentel baseline — PPG-DaLiA:", dalia_results)


WESAD X: (1797, 2048) | y: (1797,) | sid: (1797,)
DaLiA X: (3883, 2048) | y: (3883,) | sid: (3883,)
WESAD unique patient_id: [ 2  3  4  5  6  7  8  9 10 11]
DaLiA unique patient_id: [ 1  2  3  4  5  7  8  9 10 11 12 13 14 15]
Pimentel baseline — WESAD: {'N_test': 1797, 'coverage': 0.8491930996104619, 'MAE': 9.790989978042347, 'RMSE': 11.562490032164419, 'bias': -7.825107842626697, 'LoA_low': -24.51464042737444, 'LoA_high': 8.864424742121049}
Pimentel baseline — PPG-DaLiA: {'N_test': 3883, 'coverage': 0.8799896986865825, 'MAE': 13.290223961682338, 'RMSE': 18.242279414445882, 'bias': 0.5664499755025886, 'LoA_low': -35.17640672017474, 'LoA_high': 36.30930667117991}


In [8]:
def eval_one_subject_windows(
    X_ppg: np.ndarray,
    y_rr: np.ndarray,
    sid: np.ndarray,
    subject_id: int,
    cfg: PimentelConfig,
    dataset_name: str
) -> pd.DataFrame:
    """
    Returns a per-window DataFrame for ONE subject:
    columns: dataset, subject_id, idx, rr_ref, rr_pred, sqi, is_valid
    """
    sid = np.asarray(sid).astype(int)
    y_rr = np.asarray(y_rr).astype(float)

    mask = (sid == int(subject_id))
    X_s = X_ppg[mask]
    y_s = y_rr[mask]

    rr_pred = np.full(len(y_s), np.nan, dtype=float)
    sqi_arr = np.full(len(y_s), np.nan, dtype=float)

    for i in range(len(y_s)):
        rr_hat, sqi = estimate_rr_pimentel_window(X_s[i], cfg)
        sqi_arr[i] = sqi
        if rr_hat is not None:
            rr_pred[i] = rr_hat

    out = pd.DataFrame({
        "dataset": dataset_name,
        "subject_id": int(subject_id),
        "idx_in_subject": np.arange(len(y_s), dtype=int),
        "rr_ref": y_s,
        "rr_pred": rr_pred,
        "sqi": sqi_arr,
    })
    out["is_valid"] = np.isfinite(out["rr_pred"]).astype(int)
    return out


NameError: name 'PimentelConfig' is not defined

In [18]:
def summarize_by_subject(per_window_df: pd.DataFrame) -> pd.DataFrame:
    """
    Produces per-subject MAE/RMSE/coverage from per-window data.
    """
    rows = []
    for (ds, s), g in per_window_df.groupby(["dataset", "subject_id"]):
        rr_ref = g["rr_ref"].to_numpy(float)
        rr_pred = g["rr_pred"].to_numpy(float)

        coverage = float(np.mean(np.isfinite(rr_pred)))
        mae, rmse = mae_rmse(rr_ref, rr_pred)   # uses your existing function

        rows.append({
            "dataset": ds,
            "subject_id": int(s),
            "N": int(len(g)),
            "coverage": coverage,
            "MAE": mae,
            "RMSE": rmse
        })
    return pd.DataFrame(rows)

# ---- run LOSO-style eval (really: per-subject eval) ----
wesad_subjects = np.unique(np.asarray(wesad_sid).astype(int))
dalia_subjects = np.unique(np.asarray(dalia_sid).astype(int))

wesad_rows = []
for s in wesad_subjects:
    wesad_rows.append(eval_one_subject_windows(wesad_X, wesad_rr, wesad_sid, s, cfg, "WESAD"))

dalia_rows = []
for s in dalia_subjects:
    dalia_rows.append(eval_one_subject_windows(dalia_X, dalia_rr, dalia_sid, s, cfg, "PPG-DaLiA"))

per_window = pd.concat(wesad_rows + dalia_rows, ignore_index=True)

# Per-subject summary
per_subject = summarize_by_subject(per_window)

print("\nPer-subject results:")
display(per_subject.sort_values(["dataset", "subject_id"]))

print("\nAggregate across subjects (mean):")
display(per_subject.groupby("dataset")[["coverage", "MAE", "RMSE"]].mean())

print("\nAggregate across subjects (median):")
display(per_subject.groupby("dataset")[["coverage", "MAE", "RMSE"]].median())



Per-subject results:


,dataset,subject_id,N,coverage,MAE,RMSE
0,PPG-DaLiA,1,288,0.836806,10.535065,14.179888
1,PPG-DaLiA,2,256,0.964844,10.696660,14.326193
2,PPG-DaLiA,3,273,0.912088,14.007364,19.206738
3,PPG-DaLiA,4,286,0.972028,10.838227,14.015652
4,PPG-DaLiA,5,291,1.000000,11.575311,13.432429
5,PPG-DaLiA,7,292,0.931507,10.066455,12.950637
6,PPG-DaLiA,8,252,0.980159,16.015961,22.323969
7,PPG-DaLiA,9,268,0.906716,16.165173,22.564109
8,PPG-DaLiA,10,333,0.915916,13.261512,18.008121
9,PPG-DaLiA,11,283,0.989399,9.507142,13.816488



Aggregate across subjects (mean):


,coverage,MAE,RMSE
dataset,,,
PPG-DaLiA,0.875619,13.416338,17.815475
WESAD,0.853499,9.696241,11.233435



Aggregate across subjects (median):


,coverage,MAE,RMSE
dataset,,,
PPG-DaLiA,0.923711,12.418411,16.775573
WESAD,0.860031,9.647503,10.789534


In [19]:
import numpy as np

def print_window_assumptions(name, X, rr, sid, fs=64.0):
    X = np.asarray(X)
    rr = np.asarray(rr)
    sid = np.asarray(sid)

    print(f"\n=== {name} windowing checks ===")
    print("X shape:", X.shape)
    print("rr shape:", rr.shape, "sid shape:", sid.shape)
    print("fs:", fs, "Hz")
    print("window seconds:", X.shape[1] / fs)
    print("unique subjects:", np.unique(sid.astype(int)))
    assert X.ndim == 2, "X must be (N,L)"
    assert X.shape[0] == len(rr) == len(sid), "X/y/sid must align 1:1"
    assert abs(X.shape[1]/fs - 32.0) < 1e-6, "Expected 32s windows"
    print("✅ PASS")

print_window_assumptions("WESAD", wesad_X, wesad_rr, wesad_sid, fs=64.0)
print_window_assumptions("PPG-DaLiA", dalia_X, dalia_rr, dalia_sid, fs=64.0)



=== WESAD windowing checks ===
X shape: (1797, 2048)
rr shape: (1797,) sid shape: (1797,)
fs: 64.0 Hz
window seconds: 32.0
unique subjects: [ 2  3  4  5  6  7  8  9 10 11]
✅ PASS

=== PPG-DaLiA windowing checks ===
X shape: (3883, 2048)
rr shape: (3883,) sid shape: (3883,)
fs: 64.0 Hz
window seconds: 32.0
unique subjects: [ 1  2  3  4  5  7  8  9 10 11 12 13 14 15]
✅ PASS


In [20]:
import pandas as pd
import numpy as np

def run_baseline_per_window(X, rr, sid, cfg, dataset_name):
    X = np.asarray(X)
    rr = np.asarray(rr).astype(float)
    sid = np.asarray(sid).astype(int)

    rows = []
    for i in range(len(rr)):
        rr_hat, sqi = estimate_rr_pimentel_window(X[i], cfg)  # uses your baseline
        rows.append({
            "dataset": dataset_name,
            "window_index": i,
            "subject_id": int(sid[i]),
            "rr_ref": float(rr[i]),
            "rr_pred": (float(rr_hat) if rr_hat is not None else np.nan),
            "sqi": float(sqi) if np.isfinite(sqi) else np.nan,
            "is_valid": bool(rr_hat is not None),
        })
    return pd.DataFrame(rows)

wesad_df = run_baseline_per_window(wesad_X, wesad_rr, wesad_sid, cfg, "WESAD")
dalia_df = run_baseline_per_window(dalia_X, dalia_rr, dalia_sid, cfg, "PPG-DaLiA")

print(wesad_df.head())
print(dalia_df.head())


  dataset  window_index  subject_id     rr_ref    rr_pred       sqi  is_valid
0   WESAD             0           2  16.153846        NaN  0.803589     False
1   WESAD             1           2  22.641509        NaN  0.541797     False
2   WESAD             2           2  24.444444   4.218750  0.947211      True
3   WESAD             3           2  24.000000   4.218750  0.947504      True
4   WESAD             4           2  24.220183  17.578125  0.960648      True
     dataset  window_index  subject_id     rr_ref    rr_pred       sqi  \
0  PPG-DaLiA             0           1  21.052632  54.843750  0.913760   
1  PPG-DaLiA             1           1  22.018349  53.671875  0.904785   
2  PPG-DaLiA             2           1  14.358974        NaN  0.898558   
3  PPG-DaLiA             3           1  11.764706        NaN  0.834961   
4  PPG-DaLiA             4           1  14.608696        NaN  0.820312   

   is_valid  
0      True  
1      True  
2     False  
3     False  
4     False  


In [21]:
import numpy as np
import pandas as pd

def mae_rmse_from_df(df):
    d = df.dropna(subset=["rr_pred"]).copy()
    if len(d) == 0:
        return np.nan, np.nan, 0.0
    err = d["rr_pred"].to_numpy() - d["rr_ref"].to_numpy()
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    coverage = float(len(d) / len(df))
    return mae, rmse, coverage

def per_subject_summary(per_window_df):
    out = []
    for s, g in per_window_df.groupby("subject_id"):
        mae, rmse, cov = mae_rmse_from_df(g)
        out.append({"subject_id": int(s), "MAE": mae, "RMSE": rmse, "coverage": cov, "N": int(len(g))})
    return pd.DataFrame(out).sort_values("subject_id")

wesad_subj = per_subject_summary(wesad_df)
dalia_subj = per_subject_summary(dalia_df)

print("\nWESAD per-subject:\n", wesad_subj)
print("\nDaLiA per-subject:\n", dalia_subj)

def report_across_subjects(name, subj_df):
    print(f"\n=== {name} across-subject report ===")
    print("MAE mean:", float(subj_df["MAE"].mean()), " | median:", float(subj_df["MAE"].median()))
    print("RMSE mean:", float(subj_df["RMSE"].mean()), " | median:", float(subj_df["RMSE"].median()))
    print("Coverage mean:", float(subj_df["coverage"].mean()), " | median:", float(subj_df["coverage"].median()))

report_across_subjects("WESAD", wesad_subj)
report_across_subjects("PPG-DaLiA", dalia_subj)



WESAD per-subject:
    subject_id        MAE       RMSE  coverage    N
0           2  13.005225  14.504626  0.844086  186
1           3   8.543998  10.018191  0.868020  197
2           4   9.958239  11.140778  0.852041  196
3           5   8.919227  10.198030  0.931579  190
4           6   9.829724  11.396643  0.675926  216
5           7  11.789016  16.020486  0.968750  160
6           8   8.656590   9.728204  0.688623  167
7           9   5.984329   7.280075  0.765823  158
8          10  10.810782  11.609025  0.946429  168
9          11   9.465282  10.438289  0.993711  159

DaLiA per-subject:
     subject_id        MAE       RMSE  coverage    N
0            1  10.535065  14.179888  0.836806  288
1            2  10.696660  14.326193  0.964844  256
2            3  14.007364  19.206738  0.912088  273
3            4  10.838227  14.015652  0.972028  286
4            5  11.575311  13.432429  1.000000  291
5            7  10.066455  12.950637  0.931507  292
6            8  16.015961  22.323

In [22]:
import json
import numpy as np

def make_loso_folds(subject_ids):
    subject_ids = sorted([int(s) for s in np.unique(subject_ids)])
    folds = []
    for s in subject_ids:
        folds.append({
            "test_subjects": [s],
            "train_subjects": [x for x in subject_ids if x != s]
        })
    return folds

splits = {
    "protocol": "LOSO",
    "WESAD_subjects": sorted([int(s) for s in np.unique(wesad_sid)]),
    "PPG_DaLiA_subjects": sorted([int(s) for s in np.unique(dalia_sid)]),
    "folds_WESAD": make_loso_folds(wesad_sid),
    "folds_PPG_DaLiA": make_loso_folds(dalia_sid),
}

with open(os.path.join(out_dir, "splits_loso.json"), "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=2)

print("Saved:", os.path.join(out_dir, "splits_loso.json"))


NameError: name 'out_dir' is not defined

In [ ]:
import json
import numpy as np

def make_loso_folds(subject_ids):
    subject_ids = sorted([int(s) for s in np.unique(subject_ids)])
    folds = []
    for s in subject_ids:
        folds.append({
            "test_subjects": [s],
            "train_subjects": [x for x in subject_ids if x != s]
        })
    return folds

splits = {
    "protocol": "LOSO",
    "WESAD_subjects": sorted([int(s) for s in np.unique(wesad_sid)]),
    "PPG_DaLiA_subjects": sorted([int(s) for s in np.unique(dalia_sid)]),
    "folds_WESAD": make_loso_folds(wesad_sid),
    "folds_PPG_DaLiA": make_loso_folds(dalia_sid),
}

with open(os.path.join(out_dir, "splits_loso.json"), "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=2)

print("Saved:", os.path.join(out_dir, "splits_loso.json"))


In [ ]:
# =========================
# SAVE PIMENTEL BASELINE (robust, auto-detects variable names)
# =========================
import numpy as np
import pandas as pd

def _pick_first_existing(names, g):
    for n in names:
        if n in g and g[n] is not None:
            return n, g[n]
    return None, None

def _per_subject_from_per_window(df):
    # expects: subject_id, rr_ref, rr_pred
    rows = []
    for s, g in df.groupby("subject_id"):
        y = g["rr_ref"].to_numpy(float)
        p = g["rr_pred"].to_numpy(float)
        m = np.isfinite(y) & np.isfinite(p)
        mae = float(np.mean(np.abs(p[m] - y[m]))) if np.any(m) else np.nan
        rmse = float(np.sqrt(np.mean((p[m] - y[m])**2))) if np.any(m) else np.nan
        rows.append({"subject_id": int(s), "N": int(len(g)), "MAE": mae, "RMSE": rmse})
    return pd.DataFrame(rows).sort_values("subject_id").reset_index(drop=True)

g = globals()

# Common names you might have in your baseline notebook:
WESAD_PW_CANDIDATES = ["wesad_df", "wesad_all", "wesad_pw", "WESAD_df", "df_wesad"]
WESAD_PS_CANDIDATES = ["wesad_subj", "wesad_sub", "wesad_ps", "WESAD_subj", "dfsub_wesad"]

DALIA_PW_CANDIDATES = ["dalia_df", "dalia_all", "dalia_pw", "PPGDalia_df", "df_dalia"]
DALIA_PS_CANDIDATES = ["dalia_subj", "dalia_sub", "dalia_ps", "PPGDalia_subj", "dfsub_dalia"]

wesad_pw_name, wesad_pw = _pick_first_existing(WESAD_PW_CANDIDATES, g)
wesad_ps_name, wesad_ps = _pick_first_existing(WESAD_PS_CANDIDATES, g)

dalia_pw_name, dalia_pw = _pick_first_existing(DALIA_PW_CANDIDATES, g)
dalia_ps_name, dalia_ps = _pick_first_existing(DALIA_PS_CANDIDATES, g)

print("Detected WESAD per-window var:", wesad_pw_name)
print("Detected WESAD per-subject var:", wesad_ps_name)
print("Detected DaLiA per-window var:", dalia_pw_name)
print("Detected DaLiA per-subject var:", dalia_ps_name)

# If per-subject missing but per-window exists, compute per-subject
if wesad_pw is not None and wesad_ps is None:
    wesad_ps = _per_subject_from_per_window(wesad_pw)

if dalia_pw is not None and dalia_ps is None:
    dalia_ps = _per_subject_from_per_window(dalia_pw)

# Hard requirements
missing = []
if wesad_pw is None: missing.append("WESAD per-window dataframe (need columns: subject_id, rr_ref, rr_pred, window_index)")
if dalia_pw is None: missing.append("DaLiA per-window dataframe (need columns: subject_id, rr_ref, rr_pred, window_index)")

if missing:
    print("\n❌ Nothing saved because these are missing:")
    for m in missing:
        print(" -", m)
    print("\nFix: run the training/evaluation cells ABOVE this until you see per-window outputs, then run this save cell again.")
    raise NameError("Missing per-window outputs in this notebook session.")

# Ensure required columns exist
required_cols = {"subject_id", "rr_ref", "rr_pred"}
for name, df in [("wesad_pw", wesad_pw), ("dalia_pw", dalia_pw)]:
    if not required_cols.issubset(set(df.columns)):
        raise ValueError(f"{name} is missing required columns {required_cols}. Found: {list(df.columns)}")

# Config
pimentel_config = {
    "method": "pimentel_baseline",
    "notes": "PPG-only baseline (Pimentel). Saved from current notebook variables.",
}

# Save
run_dir_w = make_run_dir("pimentel_baseline", "WESAD", tag="ppg_only")
save_run(run_dir_w, wesad_pw, wesad_ps, pimentel_config, splits_path="splits_loso.json")

run_dir_d = make_run_dir("pimentel_baseline", "PPG-DaLiA", tag="ppg_only")
save_run(run_dir_d, dalia_pw, dalia_ps, pimentel_config, splits_path="splits_loso.json")


In [24]:
# =========================
# Detect baseline outputs + save cleanly (Pimentel baseline)
# =========================
import os, json, shutil
from datetime import datetime
import pandas as pd
import numpy as np

RUNS_ROOT = r"C:\Users\yasmi\rr_runs"
METHOD_ROOT = "pimentel_baseline"
os.makedirs(RUNS_ROOT, exist_ok=True)

def make_run_dir(method, dataset, tag="ppg_only"):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    d = os.path.join(RUNS_ROOT, method, dataset, f"{ts}_{tag}")
    os.makedirs(d, exist_ok=True)
    return d

def save_run(run_dir, per_window_df, per_subject_df, config_dict, splits_path=None):
    per_window_df.to_csv(os.path.join(run_dir, "per_window.csv"), index=False)
    per_subject_df.to_csv(os.path.join(run_dir, "per_subject.csv"), index=False)
    with open(os.path.join(run_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config_dict, f, indent=2)
    if splits_path is not None and os.path.exists(splits_path):
        shutil.copy2(splits_path, os.path.join(run_dir, "splits_loso.json"))
    print(f"\n✅ Saved run to: {run_dir}")
    print(f"  - per_window.csv | rows: {len(per_window_df)}")
    print(f"  - per_subject.csv | rows: {len(per_subject_df)}")

def _per_subject_from_per_window(df):
    df = df.copy()
    df["abs_err"] = np.abs(df["rr_pred"].astype(float) - df["rr_ref"].astype(float))
    rows = []
    for s, g in df.groupby("subject_id"):
        g_valid = g[g["is_valid"] == True]
        if len(g_valid) == 0:
            rows.append({"subject_id": int(s), "MAE": np.nan, "RMSE": np.nan, "N": 0})
        else:
            e = g_valid["rr_pred"].astype(float).values - g_valid["rr_ref"].astype(float).values
            rows.append({
                "subject_id": int(s),
                "MAE": float(np.mean(np.abs(e))),
                "RMSE": float(np.sqrt(np.mean(e * e))),
                "N": int(len(g_valid)),
            })
    return pd.DataFrame(rows).sort_values("subject_id").reset_index(drop=True)

# ---- Detect variables safely ----
wesad_df_var = None
dalia_df_var = None

for cand in ["wesad_df", "wesad_all", "df_wesad", "wesad_per_window"]:
    if cand in globals() and isinstance(globals()[cand], pd.DataFrame):
        wesad_df_var = cand
        break

for cand in ["dalia_df", "dalia_all", "df_dalia", "dalia_per_window"]:
    if cand in globals() and isinstance(globals()[cand], pd.DataFrame):
        dalia_df_var = cand
        break

print("Detected WESAD per-window var:", wesad_df_var)
print("Detected DaLiA per-window var:", dalia_df_var)

if wesad_df_var is None:
    raise NameError("I expected a WESAD per-window DataFrame (e.g., wesad_df) to exist, but none was found.")
if dalia_df_var is None:
    raise NameError("I expected a DaLiA per-window DataFrame (e.g., dalia_df) to exist, but none was found.")

wesad_df = globals()[wesad_df_var].copy()
dalia_df = globals()[dalia_df_var].copy()

# ---- Ensure required columns exist ----
need_cols = {"window_index", "subject_id", "rr_ref", "rr_pred", "is_valid"}
missing_w = need_cols - set(wesad_df.columns)
missing_d = need_cols - set(dalia_df.columns)
if missing_w:
    raise ValueError(f"WESAD per-window df missing columns: {sorted(list(missing_w))}")
if missing_d:
    raise ValueError(f"DaLiA per-window df missing columns: {sorted(list(missing_d))}")

# ---- Build per-subject ----
wesad_subj = _per_subject_from_per_window(wesad_df)
dalia_subj = _per_subject_from_per_window(dalia_df)

# ---- Save ----
pimentel_config = {
    "method": "pimentel_baseline",
    "note": "PPG-only classical baseline (per-window + per-subject computed here)"
}

run_dir_w = make_run_dir(METHOD_ROOT, "WESAD", tag="ppg_only")
save_run(run_dir_w, wesad_df, wesad_subj, pimentel_config, splits_path=SPLITS_JSON if "SPLITS_JSON" in globals() else None)

run_dir_d = make_run_dir(METHOD_ROOT, "PPG-DaLiA", tag="ppg_only")
save_run(run_dir_d, dalia_df, dalia_subj, pimentel_config, splits_path=SPLITS_JSON if "SPLITS_JSON" in globals() else None)


Detected WESAD per-window var: wesad_df
Detected DaLiA per-window var: dalia_df

✅ Saved run to: C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_222540_ppg_only
  - per_window.csv | rows: 1797
  - per_subject.csv | rows: 10

✅ Saved run to: C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_222540_ppg_only
  - per_window.csv | rows: 3883
  - per_subject.csv | rows: 14


In [25]:
print("WESAD subjects:", sorted(wesad_df["subject_id"].unique().tolist()))
print("DaLiA subjects:", sorted(dalia_df["subject_id"].unique().tolist()))
print("WESAD rows:", len(wesad_df), "valid:", wesad_df["is_valid"].mean())
print("DaLiA rows:", len(dalia_df), "valid:", dalia_df["is_valid"].mean())


WESAD subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
DaLiA subjects: [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]
WESAD rows: 1797 valid: 0.8491930996104619
DaLiA rows: 3883 valid: 0.8799896986865825


In [1]:
import sys, platform, numpy as np, pandas as pd
try:
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torchv = torch.__version__
except Exception:
    dev = "unknown"
    torchv = "unknown"

print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torchv)
print("device:", dev)


python: 3.13.5
platform: Windows-11-10.0.26100-SP0
numpy: 2.1.3
pandas: 2.2.3
torch: 2.9.1+cpu
device: cpu


In [2]:
import json, os, numpy as np

print("has splits in memory:", "splits" in globals())
if "SPLITS_JSON" in globals():
    print("SPLITS_JSON:", SPLITS_JSON, "| exists:", os.path.exists(SPLITS_JSON))

# show splits keys if present
if "splits" in globals():
    print("splits keys:", list(splits.keys()))

# quick fold peek (works with anyformat iterator)
def _peek(dataset):
    folds = list(iter_folds_from_splits_anyformat(splits, dataset))
    print("\n===", dataset, "===")
    print("n_folds:", len(folds))
    print("fold1 keys:", folds[0].keys())
    print("fold1 train:", folds[0]["train"])
    print("fold1 test :", folds[0]["test"])
    return folds

_ = _peek("WESAD")
_ = _peek("PPG-DaLiA")


has splits in memory: False


NameError: name 'iter_folds_from_splits_anyformat' is not defined

In [9]:
from pathlib import Path

RUNS_ROOT = Path.home() / "rr_runs"
print("RUNS_ROOT:", RUNS_ROOT, "| exists:", RUNS_ROOT.exists())

def scan(method, dataset):
    base = RUNS_ROOT / method / dataset
    print(f"\n=== scan: {method} / {dataset} ===")
    print("base exists:", base.exists(), "| path:", base)
    if not base.exists():
        return
    runs = sorted([p for p in base.iterdir() if p.is_dir()], reverse=True)
    print("n_runs:", len(runs))
    for p in runs[:10]:
        files = {f.name for f in p.iterdir() if f.is_file()}
        flags = []
        if "per_window.csv" in files and "per_subject.csv" in files: flags.append("FINAL")
        if "per_window_partial.csv" in files and "per_subject_partial.csv" in files: flags.append("PARTIAL")
        if "config.json" in files: flags.append("CFG")
        if "splits_loso.json" in files: flags.append("SPLITS_COPY")
        print(" -", p.name, "|", ",".join(flags) if flags else "(no csv)")

scan("kazemi_strong", "WESAD")
scan("kazemi_strong", "PPG-DaLiA")

scan("proposed_v6_tuned", "WESAD")
scan("proposed_v6_tuned", "PPG-DaLiA")

scan("pimentel_baseline", "WESAD")
scan("pimentel_baseline", "PPG-DaLiA")


RUNS_ROOT: C:\Users\yasmi\rr_runs | exists: True

=== scan: kazemi_strong / WESAD ===
base exists: True | path: C:\Users\yasmi\rr_runs\kazemi_strong\WESAD
n_runs: 1
 - 20260126_195201 | FINAL,CFG,SPLITS_COPY

=== scan: kazemi_strong / PPG-DaLiA ===
base exists: True | path: C:\Users\yasmi\rr_runs\kazemi_strong\PPG-DaLiA
n_runs: 1
 - 20260126_195202 | FINAL,CFG,SPLITS_COPY

=== scan: proposed_v6_tuned / WESAD ===
base exists: True | path: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD
n_runs: 9
 - 20260128_133616_ckpt_detval | (no csv)
 - 20260128_111913_ckpt_detval | PARTIAL
 - 20260128_105330_ckpt_detval | (no csv)
 - 20260127_133744 | (no csv)
 - 20260127_124555 | (no csv)
 - 20260127_124526 | (no csv)
 - 20260126_201212 | FINAL,CFG,SPLITS_COPY
 - 20260126_195354 | (no csv)
 - 20260126_195331 | (no csv)

=== scan: proposed_v6_tuned / PPG-DaLiA ===
base exists: True | path: C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA
n_runs: 1
 - 20260128_111913_ckpt_detval | PARTIAL

=== scan:

In [11]:
import os, json, shutil
import pandas as pd

SPLITS_JSON = r"C:\Jupyter Files\baseline_outputs\splits_loso.json"
assert os.path.exists(SPLITS_JSON), f"Missing SPLITS_JSON: {SPLITS_JSON}"

PROPOSED_WESAD_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval"
PROPOSED_DALIA_DIR = r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval"

def _safe_write_csv(df, path):
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def finalize_partials(run_dir):
    pw_part = os.path.join(run_dir, "per_window_partial.csv")
    ps_part = os.path.join(run_dir, "per_subject_partial.csv")
    pw_out  = os.path.join(run_dir, "per_window.csv")
    ps_out  = os.path.join(run_dir, "per_subject.csv")

    print("\n=== Finalize ===")
    print("run_dir:", run_dir)
    assert os.path.exists(run_dir), "Run dir does not exist"

    # Copy splits (optional but nice to keep consistent)
    dst_splits = os.path.join(run_dir, "splits_loso.json")
    if not os.path.exists(dst_splits):
        shutil.copy2(SPLITS_JSON, dst_splits)
        print("copied splits_loso.json ✅")
    else:
        print("splits_loso.json already exists ✅")

    # Finalize per_window
    if os.path.exists(pw_part):
        dfw = pd.read_csv(pw_part)
        # light de-dup if resume ever appended duplicates
        key_cols = [c for c in ["dataset","fold","subject_id","window_index"] if c in dfw.columns]
        if len(key_cols) >= 2:
            dfw = dfw.drop_duplicates(subset=key_cols, keep="last")
        _safe_write_csv(dfw, pw_out)
        print("wrote per_window.csv ✅ | rows:", len(dfw))
    elif os.path.exists(pw_out):
        dfw = pd.read_csv(pw_out)
        print("per_window.csv already exists ✅ | rows:", len(dfw))
    else:
        raise FileNotFoundError("Neither per_window_partial.csv nor per_window.csv exists")

    # Finalize per_subject
    if os.path.exists(ps_part):
        dfs = pd.read_csv(ps_part)
        key_cols = [c for c in ["dataset","fold","subject_id"] if c in dfs.columns]
        if len(key_cols) >= 2:
            dfs = dfs.drop_duplicates(subset=key_cols, keep="last")
        _safe_write_csv(dfs, ps_out)
        print("wrote per_subject.csv ✅ | rows:", len(dfs))
    elif os.path.exists(ps_out):
        dfs = pd.read_csv(ps_out)
        print("per_subject.csv already exists ✅ | rows:", len(dfs))
    else:
        raise FileNotFoundError("Neither per_subject_partial.csv nor per_subject.csv exists")

finalize_partials(PROPOSED_WESAD_DIR)
finalize_partials(PROPOSED_DALIA_DIR)

print("\n✅ Done. Proposed det-val runs are now FINAL.")



=== Finalize ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval
copied splits_loso.json ✅
wrote per_window.csv ✅ | rows: 1797
wrote per_subject.csv ✅ | rows: 10

=== Finalize ===
run_dir: C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval
copied splits_loso.json ✅
wrote per_window.csv ✅ | rows: 3883
wrote per_subject.csv ✅ | rows: 14

✅ Done. Proposed det-val runs are now FINAL.


In [12]:
RUNS = {
  "kazemi_strong": {
    "WESAD": r"C:\Users\yasmi\rr_runs\kazemi_strong\WESAD\20260126_195201",
    "PPG-DaLiA": r"C:\Users\yasmi\rr_runs\kazemi_strong\PPG-DaLiA\20260126_195202",
  },
  "proposed_v6_tuned_detval": {
    "WESAD": r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\WESAD\20260128_111913_ckpt_detval",
    "PPG-DaLiA": r"C:\Users\yasmi\rr_runs\proposed_v6_tuned\PPG-DaLiA\20260128_111913_ckpt_detval",
  },
  "pimentel_baseline_ppg_only": {
    "WESAD": r"C:\Users\yasmi\rr_runs\pimentel_baseline\WESAD\20260126_212659_ppg_only",
    "PPG-DaLiA": r"C:\Users\yasmi\rr_runs\pimentel_baseline\PPG-DaLiA\20260126_212700_ppg_only",
  }
}

import os, pandas as pd

for method, ds_map in RUNS.items():
    print("\n===", method, "===")
    for ds, d in ds_map.items():
        pw = os.path.join(d, "per_window.csv")
        ps = os.path.join(d, "per_subject.csv")
        ok = os.path.exists(pw) and os.path.exists(ps)
        print(ds, "| ok:", ok)
        if os.path.exists(ps):
            dfs = pd.read_csv(ps)
            print("  per_subject rows:", len(dfs))
        if os.path.exists(pw):
            dfw = pd.read_csv(pw)
            print("  per_window rows:", len(dfw))



=== kazemi_strong ===
WESAD | ok: True
  per_subject rows: 10
  per_window rows: 1797
PPG-DaLiA | ok: True
  per_subject rows: 14
  per_window rows: 3883

=== proposed_v6_tuned_detval ===
WESAD | ok: True
  per_subject rows: 10
  per_window rows: 1797
PPG-DaLiA | ok: True
  per_subject rows: 14
  per_window rows: 3883

=== pimentel_baseline_ppg_only ===
WESAD | ok: True
  per_subject rows: 10
  per_window rows: 1797
PPG-DaLiA | ok: True
  per_subject rows: 14
  per_window rows: 3883
